# 标记笔记本
你可以通过运行这个笔记本，按照流程标记你的目标

In [ ]:
from pathlib import Path
import re

import numpy as np
from numpy.typing import NDArray
from scipy import ndimage as nd 
from tifffile import imread, imwrite
import napari
from loguru import logger
from tqdm import tqdm
import typer

In [ ]:
from pathlib import Path

from dotenv import load_dotenv
from loguru import logger

# Load environment variables from .env file if it exists
load_dotenv()

# Paths
PROJ_ROOT = Path("notebooks").resolve().parents[1]
logger.info(f"PROJ_ROOT path is: {PROJ_ROOT}")

DATA_DIR = PROJ_ROOT / "data"
RAW_DATA_DIR = DATA_DIR / "raw"
IMAGES_DATA_DIR = DATA_DIR / "images"
INTERIM_DATA_DIR = DATA_DIR / "interim"
PROCESSED_DATA_DIR = DATA_DIR / "processed"
EXTERNAL_DATA_DIR = DATA_DIR / "external"

TRAIN_DATA_DIR = PROCESSED_DATA_DIR / "train"
TEST_DATA_DIR = PROCESSED_DATA_DIR / "test"
MODELS_DIR = PROJ_ROOT / "models"

REPORTS_DIR = PROJ_ROOT / "reports"
FIGURES_DIR = REPORTS_DIR / "figures"

# If tqdm is installed, configure loguru with tqdm.write
# https://github.com/Delgan/loguru/issues/135
try:
    from tqdm import tqdm

    logger.remove(0)
    logger.add(lambda msg: tqdm.write(msg, end=""), colorize=True)
except ModuleNotFoundError:
    pass

In [ ]:
def roi_cut(img: NDArray, masks: NDArray, roi: NDArray | None=None):
    """
    img: 原始图像，形状 (H, W) 或 (C, H, W)
    masks: 标记掩码或识别目标，形状 (H, W) 或 (C, H, W)
    roi: 工作区域或识别区域掩码标记，形状 (H, W), roi>0 为识别区域
    返回：(img_crop, masks_crop, box)
    """
    if roi is None:
        # 全图均为识别区域，不裁剪，不置零
        img_crop = img.copy()
        masks_crop = masks.copy()
        box = (0, img.shape[-2], 0, img.shape[-1])
        return img_crop, masks_crop, box

    ys, xs = np.where(roi > 0)
    if len(ys) == 0:
        # 无识别区域，全图均不识别
        img_crop = np.zeros_like(img)
        masks_crop = np.zeros_like(masks)
        box = (0, 0, 0, 0)
        return img_crop, masks_crop, box

    # 确定裁切范围
    y0, y1 = int(ys.min()), int(ys.max()) + 1
    x0, x1 = int(xs.min()), int(xs.max()) + 1
    logger.info(f"box: y={y0}:{y1}, x={x0}:{x1}")
    roi_crop = roi[y0:y1, x0:x1]
    
    # 裁切 置零
    if img.ndim == 2:
        img_crop = img[y0:y1, x0:x1].copy()
        img_crop[roi_crop == 0] = 0
    else:
        img_crop = np.stack(
            [np.where(roi_crop == 0, 0, img[c, y0:y1, x0:x1]).astype(img.dtype) for c in range(img.shape[0])], axis=0
        )
    if masks.ndim == 2:
        masks_crop = masks[y0:y1, x0:x1].copy()
        masks_crop[roi_crop == 0] = 0
    else:
        masks_crop = np.stack(
            [np.where(roi_crop == 0, 0, masks[c, y0:y1, x0:x1]).astype(masks.dtype) for c in range(masks.shape[0])], axis=0
        )
    return img_crop, masks_crop, (y0, y1, x0, x1)

In [ ]:
# Set Path
input_path: Path = IMAGES_DATA_DIR
output_path: Path = INTERIM_DATA_DIR

In [ ]:
logger.info("Processing dataset...")
logger.info(f"Input path: {input_path}")
logger.info(f"Output path: {output_path}")
# Load image-------------------------------------
tmp_list: list[Path] = list(Path(input_path).iterdir())
img_list: list[Path] = []
for tmp in tmp_list:
    if re.search('.tif', str(tmp)):
        img_list.append(tmp) # img_list have full path, not name, e.g. "/path/to/img.tif"
        
logger.info("images found:\n"+"\n".join(map(lambda x: str(x.name), img_list)))
logger.info(f"Number of images found: {len(img_list)}")

img_index = 1

In [ ]:
# Select image and load it
logger.info(f"Selected image: {img_list[img_index].name}")
img_path: Path = img_list[img_index] # img_list have full path
logger.info(f"Image path: {img_path}")
img = imread(img_path)
logger.info(img.shape)

In [ ]:
viewer = napari.Viewer()

# 务必确定每个通道的含义，并填写指定通道序号，辅助工具：FIJI
actin_ch_number = int("3") - 1
mito_ch_number = int("1") - 1
lipid_ch_number = int("2") - 1

actin_img = img[actin_ch_number, :, :]
mito_img = img[mito_ch_number, :, :]
lipid_img = img[lipid_ch_number, :, :]

viewer.add_image(actin_img, name="actin_img")
viewer.add_image(mito_img, name="mito_img")
viewer.add_image(lipid_img, name="lipid_img")

In [ ]:
# 你需要手动标记需要识别的区域
roi_mask = viewer.layers['roi - Labels'].data

In [ ]:
# 你需要手动标记mito
mito_mask = viewer.layers['mito - Labels'].data

In [ ]:
# 你需要手动标记lipid
lipid_mask = viewer.layers['lipid - Labels'].data

In [ ]:
# Mask work area, save _masks.tif and _roi.tif in /data/external
assert mito_mask.shape == lipid_mask.shape
masks = np.stack([mito_mask, lipid_mask], axis=0).astype(np.uint16)
masks_path = EXTERNAL_DATA_DIR / f"{img_list[img_index].name.replace(".tif", "_masks.tif")}"
roi_path = EXTERNAL_DATA_DIR / f"{img_list[img_index].name.replace(".tif", "_roi.tif")}"
logger.info(f"roi_shape: {roi_mask.shape}")
logger.info(f"masks_shape: {masks.shape}")
imwrite(masks_path, np.uint16(masks))
imwrite(roi_path, np.uint16(roi_mask))
# img_crop, masks_crop, _ = roi_cut(img, masks, roi_mask)
# logger.info(f"img_crop shape: {img_crop.shape}")
# imwrite(INTERIM_DATA_DIR / f"{img_list[img_index].name.replace(".tif", "_img.tif")}", np.uint16(img_crop))
# imwrite(INTERIM_DATA_DIR / f"{img_list[img_index].name.replace(".tif", "_masks.tif")}", np.uint16(masks_crop))


In [ ]:
viewer.close()